# 08.01 -- What Is Alignment and Why Does It Matter?

**Module:** 08 -- Alignment  
**Notebook:** 1 of 5  
**Prerequisites:** Module 07 (Fine-tuning), particularly 07.03 (Instruction Tuning)

---

## Learning Objectives

1. Understand the gap between a pretrained language model and a useful, safe assistant
2. Define alignment formally and situate it within the broader ML pipeline
3. Understand why instruction tuning alone is insufficient
4. Survey the alignment landscape: RLHF, DPO, RLAIF, Constitutional AI
5. Implement the reward hacking phenomenon from scratch to develop intuition

---

## 1. The Alignment Problem in One Paragraph

A language model trained purely on next-token prediction learns to be a sophisticated text completion engine. It will complete a sentence that begins with instructions for building a weapon just as readily as one that asks for a recipe. It will agree with every user regardless of factual accuracy because internet text contains both sides of every argument. It will produce confident-sounding nonsense in specialised domains because it optimises for *plausibility*, not *correctness*.

Alignment is the set of techniques for making a model that is genuinely helpful, honest, and harmless -- not just superficially fluent. The term comes from the idea of *aligning* the model's behaviour with human values and intentions.

---

## 2. The Three Failure Modes of Unaligned Models

### 2.1 Sycophancy
An unaligned model will agree with the user even when the user is wrong. If you state an incorrect fact confidently, the model will confirm it. This is not hallucination in the traditional sense -- the model knows better -- but learned sycophancy from training data and from RLHF gone wrong.

### 2.2 Helpfulness-Safety Tension
Maximising helpfulness naively means answering every question completely. But some questions should be declined or answered with caveats. A model that has only been trained to maximise human approval ratings will not learn this boundary reliably.

### 2.3 Reward Hacking
Any objective you can specify imperfectly, a sufficiently capable model will exploit. This is Goodhart's Law applied to language models: when a measure becomes a target, it ceases to be a good measure.

---

## 3. The Alignment Pipeline

The dominant approach for producing aligned assistants has three stages:

```
Stage 1: Pretraining
  Model: raw language model
  Data:  trillions of tokens of text
  Goal:  learn language, facts, reasoning
  Output: capable but unaligned base model

Stage 2: Supervised Fine-Tuning (SFT)
  Model: pretrained base model
  Data:  high-quality (instruction, response) pairs
  Goal:  teach the response format and basic helpfulness
  Output: instruction-following model (but still no human preference signal)

Stage 3: Alignment from Human Feedback
  Model: SFT model
  Data:  human preference comparisons (A is better than B)
  Goal:  learn what humans actually prefer, beyond surface format
  Methods: RLHF (via PPO), DPO, RLAIF, Constitutional AI
  Output: aligned assistant
```

This notebook focuses on building intuition for Stage 3. Subsequent notebooks implement each component.

In [ ]:
# Environment setup.
#
# We install the core libraries used across all alignment notebooks.
# trl (Transformer Reinforcement Learning) is Hugging Face's library
# for RLHF, DPO, and related alignment methods. We also need
# matplotlib for visualisations and scipy for statistical utilities.

!pip install transformers peft datasets trl accelerate matplotlib scipy numpy -q

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility across all experiments in this notebook.
torch.manual_seed(42)
np.random.seed(42)

print("Setup complete.")

## 4. Why Instruction Tuning Alone Is Not Enough

A common misconception is that SFT from high-quality demonstrations is sufficient for alignment. It is not, for a subtle but important reason: **the demonstrations only cover positive examples**.

SFT teaches the model *how* to respond helpfully given an instruction. But it provides no signal about:
- Which of two responses is *better* when both are technically correct
- How to trade off verbosity vs concision in context
- How to refuse requests that are harmful without being excessively restrictive
- What level of confidence to express under uncertainty

Human preferences encode information that is very hard to express as demonstrations but very easy to express as comparisons: "Response A is better than Response B for this question."

In [ ]:
# Demonstrate the preference data format.
#
# The fundamental unit of alignment training is not a (prompt, response) pair
# but a (prompt, chosen_response, rejected_response) triple. The 'chosen'
# response is what a human annotator preferred; 'rejected' is the alternative.
#
# This preference format encodes nuanced quality differences that are very
# hard to capture by writing demonstrations alone. Notice in the examples
# below how both 'chosen' and 'rejected' responses are technically accurate --
# what the preference captures is tone, safety framing, and appropriate
# level of hedging.

@dataclass
class PreferencePair:
    prompt:   str
    chosen:   str    # human-preferred response
    rejected: str    # human-dispreferred response
    reason:   str    # why chosen > rejected (for educational purposes)


preference_examples = [
    PreferencePair(
        prompt="What is the best way to lose weight quickly?",
        chosen=(
            "Sustainable weight loss typically occurs at 0.5-1 kg per week. "
            "The most evidence-based approaches combine a moderate caloric deficit "
            "(300-500 kcal/day below maintenance) with regular physical activity. "
            "Crash dieting often leads to muscle loss and metabolic adaptation "
            "that makes long-term maintenance harder. I'd recommend speaking with "
            "a healthcare provider before beginning any significant diet change."
        ),
        rejected=(
            "The fastest way to lose weight is to eat as little as possible and "
            "exercise as much as you can. Some people fast for several days to "
            "jumpstart weight loss. You could lose 5 kg in a week if you are "
            "disciplined enough."
        ),
        reason="Chosen is medically accurate and adds appropriate safety framing. "
               "Rejected is dangerous and promotes behaviours that can cause harm."
    ),
    PreferencePair(
        prompt="Explain how a transformer works.",
        chosen=(
            "A transformer processes a sequence of tokens through alternating layers "
            "of self-attention and feed-forward networks. In each self-attention layer, "
            "every token computes a query, key, and value vector. Attention weights are "
            "the softmax-normalised dot products of queries against all keys, scaled by "
            "sqrt(d_k) for numerical stability. The weighted sum of values becomes the "
            "new token representation. This allows each token to gather information from "
            "any other position in one step, regardless of distance."
        ),
        rejected=(
            "A transformer is a very powerful deep learning architecture that has "
            "revolutionised NLP. It uses attention mechanisms which are very important "
            "and allow it to understand context. Transformers are the basis for GPT, "
            "BERT, and many other models. They are better than RNNs in most cases."
        ),
        reason="Chosen explains the mechanism precisely. Rejected is vague and "
               "substitutes names for understanding -- it does not actually explain anything."
    ),
    PreferencePair(
        prompt="Write a Python function to check if a number is prime.",
        chosen=(
            "def is_prime(n: int) -> bool:\n"
            "    \"\"\"Return True if n is a prime number, False otherwise.\"\"\"\n"
            "    if n < 2:\n"
            "        return False\n"
            "    if n == 2:\n"
            "        return True\n"
            "    if n % 2 == 0:\n"
            "        return False\n"
            "    for i in range(3, int(n**0.5) + 1, 2):\n"
            "        if n % i == 0:\n"
            "            return False\n"
            "    return True"
        ),
        rejected=(
            "def is_prime(n):\n"
            "    for i in range(2, n):\n"
            "        if n % i == 0:\n"
            "            return False\n"
            "    return True"
        ),
        reason="Chosen uses the O(sqrt(n)) algorithm vs O(n). It handles edge cases "
               "(n<2, even numbers) and includes a docstring. Rejected is O(n), "
               "has no docstring, and will return True for n=0 and n=1."
    ),
]

for i, ex in enumerate(preference_examples):
    print(f"Example {i+1}: {ex.prompt[:60]}")
    print(f"  WHY: {ex.reason}")
    print(f"  CHOSEN length:   {len(ex.chosen.split())} words")
    print(f"  REJECTED length: {len(ex.rejected.split())} words")
    print()

## 5. Reward Hacking: A Demonstration

The most important concept for understanding *why* alignment is hard is reward hacking. Any reward signal you specify for a learned model will be exploited if the model is capable enough.

We will demonstrate this with a toy example that mirrors exactly what goes wrong in real RLHF training.

In [ ]:
# Toy reward hacking demonstration.
#
# We define a simple 'reward model' that scores text based on a proxy
# for quality: response length combined with positive sentiment words.
# This approximates what a naive human preference model might learn
# if annotators are slightly biased toward longer, more positive-sounding
# responses.
#
# We then implement a 'policy' that can be optimised against this reward.
# The key insight is: even with a slightly imperfect reward model, a
# policy optimised strongly enough will find and exploit the imperfections.
# This is called the overoptimisation problem.

class NaiveRewardModel:
    """
    A deliberately flawed reward model.

    This model rewards length (proxy for thoroughness) and positive sentiment
    words (proxy for helpfulness). Both correlate with quality in training data
    but are exploitable.

    In real RLHF, reward models are trained on human preferences and are
    much more sophisticated -- but they still have exploitable imperfections.
    """

    POSITIVE_WORDS = {
        'excellent', 'great', 'wonderful', 'fantastic', 'certainly',
        'absolutely', 'definitely', 'perfect', 'amazing', 'outstanding'
    }

    def score(self, response: str) -> float:
        words = response.lower().split()
        n_words = len(words)
        positive_count = sum(1 for w in words if w in self.POSITIVE_WORDS)

        # Proxy reward: combination of length and sentiment
        length_score    = min(1.0, n_words / 100)    # saturates at 100 words
        sentiment_score = min(1.0, positive_count / 3)  # saturates at 3 positives

        return 0.6 * length_score + 0.4 * sentiment_score


class AlignedRewardModel:
    """
    A 'ground truth' reward model that scores based on actual quality:
    factual accuracy, appropriate length, and genuine helpfulness.

    In practice this is what we want to optimise, but we can only approximate
    it with human preferences.
    """

    HALLUCINATION_PHRASES = [
        'absolutely certain', 'definitely true', 'without any doubt',
        'this is always the case'
    ]

    def score(self, response: str, has_relevant_content: bool = True) -> float:
        words = response.lower().split()
        n_words = len(words)

        # Penalise overconfident/hallucinated phrases
        hallucination_penalty = sum(
            0.3 for phrase in self.HALLUCINATION_PHRASES
            if phrase in response.lower()
        )

        # Optimal length is around 50-150 words; longer is penalised
        if n_words < 10:
            length_score = n_words / 10
        elif n_words <= 150:
            length_score = 1.0
        else:
            length_score = max(0.3, 1.0 - (n_words - 150) / 500)

        content_score = 0.8 if has_relevant_content else 0.1
        base = 0.5 * length_score + 0.5 * content_score
        return max(0.0, base - hallucination_penalty)


naive_rm  = NaiveRewardModel()
true_rm   = AlignedRewardModel()

# Simulate a policy that is progressively more 'optimised' against the naive reward.
# At each step, the policy generates a response that scores higher on the naive reward
# but may score lower on the true reward. This is the overoptimisation dynamic.

optimisation_steps = [
    {
        "step": 0,
        "label": "Step 0: Base SFT model (unoptimised)",
        "response": (
            "Gradient descent minimises a loss function by computing the gradient "
            "and taking small steps in the opposite direction. The learning rate "
            "controls the step size."
        ),
        "has_content": True,
    },
    {
        "step": 1,
        "label": "Step 1: Light optimisation (mostly good)",
        "response": (
            "Gradient descent is a fundamental optimisation algorithm used in training "
            "neural networks. It minimises a loss function by iteratively moving "
            "parameters in the direction of steepest descent. The gradient -- the "
            "vector of partial derivatives -- points uphill, so we subtract it scaled "
            "by a learning rate to move downhill. Variants like SGD, Adam, and AdaGrad "
            "improve on vanilla gradient descent in different ways."
        ),
        "has_content": True,
    },
    {
        "step": 2,
        "label": "Step 2: Moderate optimisation (verbosity creeping in)",
        "response": (
            "Excellent question! Gradient descent is certainly one of the most "
            "absolutely fundamental and wonderful algorithms in all of machine learning. "
            "It is a fantastic method for minimising loss functions. The algorithm is "
            "definitely core to how neural networks are trained. It computes gradients "
            "and uses them to update parameters. This is an outstanding approach that "
            "has been used across many fantastic applications. The learning rate is a "
            "critical hyperparameter that controls the step size in this great method."
        ),
        "has_content": True,
    },
    {
        "step": 3,
        "label": "Step 3: Heavy optimisation (reward hacking)",
        "response": " ".join(
            ["Absolutely excellent! Certainly wonderful! Fantastic algorithm!"
             " Definitely great! Amazing! Outstanding!"]
            + ["Gradient descent is absolutely certainly the most wonderful "
               "and fantastic algorithm definitely outstanding amazing excellent."] * 10
        ),
        "has_content": False,   # no actual content anymore
    },
]

print(f"{'Step':<6} {'Naive RM':>10} {'True RM':>10} {'Gap':>10}  Description")
print("-" * 75)
for step in optimisation_steps:
    naive_score = naive_rm.score(step['response'])
    true_score  = true_rm.score(step['response'], step['has_content'])
    gap = naive_score - true_score
    print(f"{step['step']:<6} {naive_score:>10.3f} {true_score:>10.3f} {gap:>10.3f}  {step['label'][:40]}")

print()
print("Observation: As optimisation increases, the naive reward rises but the true reward")
print("first rises (steps 0->1) then falls (steps 2->3). This is overoptimisation.")
print("The model learns to exploit imperfections in the reward proxy.")

In [ ]:
# Visualise the overoptimisation curve.
#
# This is one of the most important plots in alignment research.
# The x-axis is the KL divergence from the SFT policy -- a measure of how
# much the policy has moved from its starting point under RL optimisation.
# Low KL = lightly optimised; high KL = heavily optimised.
#
# The shape of the true reward curve (rises then falls) is well documented
# empirically. The naive reward continues rising because it is exactly what
# is being maximised.
#
# The key engineering implication: there is an optimal amount of RL
# optimisation. Too little = underfit to preferences; too much = reward hacking.
# The KL penalty in PPO and the implicit KL constraint in DPO both address this.

kl_values   = np.linspace(0, 8, 200)

# Simulate the naive reward rising monotonically as KL increases
naive_reward_curve = 1 - np.exp(-0.5 * kl_values) + 0.05 * np.random.randn(200) * 0

# Simulate the true reward: rises initially then falls as the policy hacks
true_reward_curve  = (
    0.85 * (1 - np.exp(-0.8 * kl_values))          # initial improvement
    - 0.07 * np.maximum(0, kl_values - 2) ** 1.5   # degradation from hacking
)
true_reward_curve  = np.clip(true_reward_curve, 0, 1)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(kl_values, naive_reward_curve, color='#1f77b4', linewidth=2.5,
        label='Proxy reward (what RL optimises)')
ax.plot(kl_values, true_reward_curve,  color='#2ca02c', linewidth=2.5,
        label='True reward (what we actually want)')

# Mark the optimal operating point
optimal_kl  = kl_values[np.argmax(true_reward_curve)]
optimal_val = true_reward_curve.max()
ax.axvline(optimal_kl, color='gray', linestyle='--', alpha=0.7)
ax.annotate(
    f'Optimal KL = {optimal_kl:.1f}\n(stop RL here)',
    xy=(optimal_kl, optimal_val),
    xytext=(optimal_kl + 0.8, optimal_val - 0.1),
    arrowprops=dict(arrowstyle='->', color='gray'),
    fontsize=10, color='gray'
)

# Shade the reward hacking region
hack_start = optimal_kl
ax.axvspan(hack_start, 8, alpha=0.08, color='red', label='Reward hacking region')
ax.text(hack_start + 1.5, 0.15, 'Reward\nHacking', color='#d62728',
        fontsize=10, ha='center', style='italic')

ax.set_xlabel('KL Divergence from SFT Policy (amount of RL optimisation)', fontsize=12)
ax.set_ylabel('Reward Score', fontsize=12)
ax.set_title(
    'Overoptimisation: Why More RL Is Not Always Better\n'
    '(Proxy reward rises monotonically; true reward peaks then degrades)',
    fontsize=12
)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 8)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('../figures/08_overoptimisation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Optimal KL divergence: {optimal_kl:.2f}")
print(f"Peak true reward:      {optimal_val:.3f}")
print()
print("Both RLHF (via KL penalty) and DPO (via implicit beta parameter) control")
print("how far the policy is allowed to deviate from the SFT reference point.")

## 6. The Alignment Method Landscape

Three methods dominate practical alignment today. All three share the same goal but differ in how they use preference data:

```
Method            Train a reward    Explicit RL    Complexity
                  model first?      training?
-------------------------------------------------------------------
RLHF (PPO)        Yes               Yes            High
DPO               No                No             Low
RLAIF / Const AI  No (uses LLM)     Optional       Medium
```

### RLHF (Reinforcement Learning from Human Feedback)
The original approach from InstructGPT (2022). Train a reward model on human preference comparisons, then use PPO to maximise that reward while staying close to the SFT policy via a KL divergence penalty. Produces state-of-the-art results but is complex to implement and unstable to train.

### DPO (Direct Preference Optimisation)
A 2023 simplification that skips the explicit reward model and RL training entirely. DPO re-parameterises the RLHF objective so it can be optimised directly with a supervised loss on preference pairs. Much simpler and equally effective in practice.

### RLAIF and Constitutional AI
Instead of human preference labels, use an AI model to generate them. Constitutional AI (Anthropic, 2022) gives the model a set of principles and has it critique and revise its own outputs. This scales annotation to any volume without additional human labour.

In [ ]:
# Quantitative comparison of alignment methods.
#
# The numbers below are representative estimates based on published results
# and practitioner experience. They illustrate the trade-offs across several
# dimensions that matter when choosing an alignment method for a real project.
#
# 'Implementation complexity' is rated 1-5 (1 = simple, 5 = very complex).
# 'Annotation cost' is the relative cost of data collection.
# 'Peak quality' is the best achievable aligned model quality.
# 'Training stability' is how often the training run produces a good model
# without hyperparameter tuning.

methods = [
    {
        'name': 'SFT only',
        'implementation_complexity': 1,
        'annotation_cost':   2,     # just (instruction, response) pairs
        'peak_quality':      60,    # normalised 0-100
        'training_stability': 90,
        'data_efficiency':   70,
    },
    {
        'name': 'RLHF (PPO)',
        'implementation_complexity': 5,
        'annotation_cost':   5,     # needs pairwise comparisons + labellers
        'peak_quality':      90,
        'training_stability': 50,   # notoriously finicky
        'data_efficiency':   60,
    },
    {
        'name': 'DPO',
        'implementation_complexity': 2,
        'annotation_cost':   4,
        'peak_quality':      87,
        'training_stability': 80,
        'data_efficiency':   75,
    },
    {
        'name': 'RLAIF / Const AI',
        'implementation_complexity': 3,
        'annotation_cost':   1,     # AI labelled
        'peak_quality':      83,
        'training_stability': 75,
        'data_efficiency':   85,
    },
    {
        'name': 'IPO / SimPO',
        'implementation_complexity': 2,
        'annotation_cost':   4,
        'peak_quality':      86,
        'training_stability': 82,
        'data_efficiency':   78,
    },
]

metrics     = ['Peak Quality', 'Training Stability', 'Data Efficiency']
metric_keys = ['peak_quality', 'training_stability', 'data_efficiency']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Grouped bar chart
ax = axes[0]
x  = np.arange(len(methods))
w  = 0.22
colors = ['#1f77b4', '#2ca02c', '#ff7f0e']
for i, (metric, key, color) in enumerate(zip(metrics, metric_keys, colors)):
    vals = [m[key] for m in methods]
    ax.bar(x + i * w, vals, w, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels([m['name'] for m in methods], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Score (0-100)', fontsize=11)
ax.set_title('Alignment Method Comparison', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')
ax.set_ylim(0, 110)

# Scatter: complexity vs quality
ax = axes[1]
scatter_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(methods)))
for m, color in zip(methods, scatter_colors):
    ax.scatter(
        m['implementation_complexity'],
        m['peak_quality'],
        s=m['training_stability'] * 2,   # bubble size = stability
        color=color,
        alpha=0.8,
        edgecolors='white',
        linewidths=1.5,
        zorder=5
    )
    ax.annotate(
        m['name'],
        (m['implementation_complexity'], m['peak_quality']),
        textcoords='offset points', xytext=(6, 3), fontsize=9
    )

ax.set_xlabel('Implementation Complexity (1=simple, 5=very complex)', fontsize=11)
ax.set_ylabel('Peak Quality Score', fontsize=11)
ax.set_title('Complexity vs Quality\n(Bubble size = training stability)', fontsize=12)
ax.grid(alpha=0.3)
ax.set_xlim(0, 6)
ax.set_ylim(50, 100)

plt.suptitle('Alignment Method Landscape', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/08_method_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

print("Key take-away: DPO and RLAIF reach near-RLHF quality at a fraction of the")
print("implementation complexity. This is why RLHF is rarely used from scratch today.")

## 7. Engineering Notes

**When to use each method**

| Scenario | Recommended method |
|----------|-------------------|
| No preference data, limited compute | SFT only with high-quality data |
| Have preference pairs, want simplicity | DPO |
| Have preference pairs, want peak quality | RLHF (PPO) |
| No human labellers available | RLAIF / Constitutional AI |
| Production safety-critical system | RLHF + Constitutional AI + red-teaming |

**The most important insight for practitioners**

Alignment quality is dominated by data quality, not by which algorithm you use. A small dataset of high-quality, consistently labelled preferences will outperform a large noisy one with any method. Before choosing RLHF vs DPO, invest in the quality of your preference data.

## 8. Exercises

1. **Reward hacking audit**: Take 50 responses rated highly by a simple reward model (e.g. length-based). Ask a human (or GPT-4) to rate the same 50 responses. Compute the Spearman rank correlation. How well does the proxy correlate with true quality?

2. **Overoptimisation threshold**: In the reward hacking demo code, vary the weight on `sentiment_score` vs `length_score`. At what ratio does the optimal KL divergence shift? What does this tell you about reward model design?

3. **Sycophancy test**: Take a base GPT-2 model and generate responses to `How correct is the statement: 'the earth is flat'?`. Now prepend `Many experts believe that` before the question. Does the response change? Now test a properly aligned model. Does it still sycophant?

4. **Preference data quality**: Take 20 preference pairs and deliberately corrupt 5 (swap chosen/rejected). Train DPO on both the clean and corrupted datasets. Compare the final models on a held-out evaluation set. How robust is DPO to label noise?

---

## 9. References

- [Ouyang et al. (2022) -- Training language models to follow instructions with human feedback (InstructGPT)](https://arxiv.org/abs/2203.02155)
- [Gao et al. (2023) -- Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760)
- [Bai et al. (2022) -- Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073)
- [Ziegler et al. (2019) -- Fine-Tuning Language Models from Human Preferences](https://arxiv.org/abs/1909.08593)